### Links and Resources
| Concept | Description |
|---|---|
| [`pandas_udf`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html) | Decorator for vectorized UDFs |
| [Pandas Function APIs](https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html) | `applyInPandas`, `mapInPandas`, `applyInPandasWithState` |
| [Apache Arrow](https://arrow.apache.org/) | Columnar interchange between JVM ↔ Python |
| [PySpark Arrow guide](https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html) | Tuning + caveats |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Pandas UDFs (Vectorized)

A Pandas UDF receives a **batch** of rows as a `pd.Series` (or `pd.DataFrame`) and returns the same. Spark uses **Apache Arrow** to move data zero-copy between the JVM and Python, then your function runs on a real numpy array, not row-by-row Python objects.

End result: Pandas UDFs are typically **10–100× faster** than plain Python UDFs and within 2–3× of pure Spark built-ins. They're the right choice for:

- Custom math that can be expressed in numpy / pandas / scipy.
- Calling sklearn / xgboost / hugging-face models on every row.
- Anything where you'd otherwise reach for a plain UDF on >1M rows.

In [ ]:
import pandas as pd
import numpy as np

df = spark.range(1_000_000).select(
    F.col("id").alias("x"),
    (F.rand(seed=1) * 100).alias("y"),
)
df.show(3)

## Three flavours

| Type | Input → Output | Use for |
|---|---|---|
| **Series → Series** (Scalar) | `pd.Series` → `pd.Series`, same length | Element-wise transforms |
| **Iterator** | `Iterator[pd.Series]` → `Iterator[pd.Series]` | Expensive setup (load model once per executor) |
| **GroupedMap / Grouped Agg** | `pd.DataFrame` per group → `pd.DataFrame` / scalar | Per-group logic that doesn't fit `groupBy.agg(...)` |

## Scalar Pandas UDF — Series → Series

The decorator's argument is the return type. Inside, you operate on the full vector — never a single value.

In [ ]:
from pyspark.sql.types import DoubleType

@F.pandas_udf(DoubleType())
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()       # vectorized — no loop

df.select("y", normalize("y").alias("y_norm")).show(5)

## Multiple input columns

In [ ]:
@F.pandas_udf(DoubleType())
def hypot(a: pd.Series, b: pd.Series) -> pd.Series:
    return pd.Series(np.hypot(a.values, b.values))

df.select(hypot("x", "y").alias("h")).show(5)

## Iterator Pandas UDF — load expensive resources once per executor

If your UDF needs an ML model, a regex compiled with a 50ms cost, or any per-call setup, the iterator form lets you initialize **once per Python worker** and reuse across batches.

In [ ]:
from typing import Iterator

@F.pandas_udf(DoubleType())
def expensive_score(it: Iterator[pd.Series]) -> Iterator[pd.Series]:
    # Imagine: model = load_my_model("s3://...")
    weights = np.array([0.5, 0.3, 0.2])   # set up once per worker
    for batch in it:
        yield pd.Series(np.minimum(batch.values * weights[0], 100.0))

df.select(expensive_score("y").alias("score")).show(5)

## Grouped Map — `applyInPandas`

When you need *one DataFrame in, one DataFrame out, per group*. Useful for per-group ML, sorting + ranking, or any logic that doesn't decompose into independent aggregates.

You must declare an **output schema** as a string or `StructType`.

In [ ]:
sales = spark.createDataFrame([
    ("EMEA", "FR", 100.0),
    ("EMEA", "FR", 150.0),
    ("EMEA", "BE", 200.0),
    ("AMER", "US", 50.0),
    ("AMER", "US", 75.0),
    ("AMER", "CA", 300.0),
] * 50, ["region", "country", "amount"])

def normalize_per_region(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf["amount_z"] = (pdf["amount"] - pdf["amount"].mean()) / pdf["amount"].std()
    return pdf

out_schema = "region string, country string, amount double, amount_z double"
sales.groupBy("region").applyInPandas(normalize_per_region, schema=out_schema).show(6)

## Grouped Aggregate — Series → scalar

Like a built-in agg (`avg`, `sum`) but with custom logic. Use `@F.pandas_udf(returnType)` and accept a `pd.Series`, return a scalar.

In [ ]:
@F.pandas_udf(DoubleType())
def trimmed_mean(s: pd.Series) -> float:
    if len(s) < 3:
        return float(s.mean())
    s = s.sort_values()
    return float(s.iloc[1:-1].mean())   # drop min and max

sales.groupBy("region").agg(
    trimmed_mean("amount").alias("trimmed_avg"),
    F.avg("amount").alias("plain_avg"),
).show()

## Map in Pandas — `mapInPandas`

For when you want batches but no grouping. Receives an iterator of `pd.DataFrame`, returns the same. Output schema declared explicitly.

In [ ]:
def add_label(it: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    for batch in it:
        batch["label"] = np.where(batch["amount"] > 100, "big", "small")
        yield batch

mapped_schema = "region string, country string, amount double, label string"
sales.mapInPandas(add_label, schema=mapped_schema).show(6)

## Memory & batch size

- **`spark.sql.execution.arrow.maxRecordsPerBatch`** (default 10,000) controls how many rows go to Python at a time. Tune down for memory-heavy UDFs, up for cheap ones.
- Each batch is a *pandas DataFrame in worker memory* — sizing matters.
- For grouped-map: **the entire group must fit in one worker**. If a region has 100M rows, this is the wrong tool; use a regular `groupBy.agg` plus higher-order functions instead.

In [ ]:
print("current batch size:", spark.conf.get("spark.sql.execution.arrow.maxRecordsPerBatch"))

## Type matching cheatsheet

The numpy/pandas dtype must match the Spark type:

| Spark | pandas |
|---|---|
| `IntegerType` | `int32` |
| `LongType` | `int64` |
| `DoubleType` | `float64` |
| `BooleanType` | `bool` |
| `StringType` | `object` (Python str) |
| `TimestampType` | `datetime64[ns]` |
| `ArrayType(T)` | `object` (list of T) |

Mismatches are a common source of `ArrowTypeError`.

## Takeaways

- Pandas UDFs are vectorized via Arrow — typically 10–100× a plain Python UDF.
- Three flavours: scalar (`Series → Series`), iterator (with reusable setup), grouped (`applyInPandas`, grouped agg, `mapInPandas`).
- Inside the function, you have full numpy/pandas/scipy/sklearn access.
- Grouped-map requires the **entire group** to fit in one worker.
- When in doubt, profile: built-in > pandas UDF > Arrow-Python UDF > plain Python UDF.